In [1]:
## ============================================
# MODEL CREATION
# ============================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import efficientnet_b2
from torchvision import models
import numpy as np
import matplotlib.pyplot as plt
from torch.cuda.amp import autocast, GradScaler

# -----------------------
# CBAM Block (Attention)
# -----------------------
class CBAMBlock(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super(CBAMBlock, self).__init__()
        # Channel Attention
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False)
        )
        self.sigmoid_channel = nn.Sigmoid()
        # Spatial Attention
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid_spatial = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()
        # Channel Attention
        avg_out = self.fc(self.avg_pool(x).view(b, c))
        max_out = self.fc(self.max_pool(x).view(b, c))
        channel_out = self.sigmoid_channel(avg_out + max_out).view(b, c, 1, 1)
        x = x * channel_out
        # Spatial Attention
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        spatial_out = torch.cat([avg_out, max_out], dim=1)
        spatial_out = self.sigmoid_spatial(self.conv_spatial(spatial_out))
        x = x * spatial_out
        return x

# ---------------------------------------------------------
# Basic Conv Block
# ---------------------------------------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel=3, dropout=0.3):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=kernel, padding=1)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout2d(dropout)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.dropout(x)
        return x


# ---------------------------------------------------------
# Residual Block (2 conv blocks + skip)
# ---------------------------------------------------------
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = ConvBlock(channels, channels)
        self.conv2 = ConvBlock(channels, channels)

    def forward(self, x):
        identity = x
        x = self.conv1(x)
        x = self.conv2(x)
        return x + identity


# ---------------------------------------------------------
# BlockTwoSkip (2 residual blocks + skip)
# ---------------------------------------------------------
class BlockTwoSkip(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.res1 = ResidualBlock(channels)
        self.res2 = ResidualBlock(channels)

    def forward(self, x):
        identity = x
        x = self.res1(x)
        x = self.res2(x)
        return x + identity

class GraphConv(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x, adj):
        # x: [B, N, C]
        # adj: [N, N]
        out = torch.matmul(adj, x)          # message passing
        out = self.linear(out)
        return F.relu(out)

def build_grid_adj(h, w, device):
    n = h * w
    adj = torch.zeros(n, n, device=device)

    for i in range(h):
        for j in range(w):
            idx = i * w + j
            for di, dj in [(-1,0),(1,0),(0,-1),(0,1)]:
                ni, nj = i+di, j+dj
                if 0 <= ni < h and 0 <= nj < w:
                    nidx = ni * w + nj
                    adj[idx, nidx] = 1

    # self-loop + normalization
    adj += torch.eye(n, device=device)
    deg = adj.sum(dim=1, keepdim=True)
    adj = adj / deg
    return adj

# ---------------------------------------------------------
# Full Model with densenet121 Backbone
# ---------------------------------------------------------
class FERModel(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()

        # -------- DenseNet Backbone --------
        backbone = models.densenet121(
            weights=models.DenseNet121_Weights.IMAGENET1K_V1
        )
        self.backbone = backbone.features   # [B, 1024, 7, 7]
        in_channels = 1024


        # Feature blocks (كما هي)
        self.conv1a = ConvBlock(in_channels, 128)
        mid_channels = 128   # 🔥 مناسب جدًا لـ JAFFE

        # -------------------
        # Feature extractor
        # -------------------
        self.conv1 = ConvBlock(in_channels, mid_channels, dropout=0.2)
        self.block1 = BlockTwoSkip(mid_channels)

        self.conv2 = ConvBlock(mid_channels, mid_channels, dropout=0.2)
        self.block2 = BlockTwoSkip(mid_channels)

        self.cbam = CBAMBlock(mid_channels)

        self.conv3 = ConvBlock(mid_channels, mid_channels, dropout=0.2)
        self.block3 = BlockTwoSkip(mid_channels)

        self.gap = nn.AdaptiveAvgPool2d(1)
        # GNN
        
        self.gnn1 = GraphConv(mid_channels, mid_channels)
        self.gnn2 = GraphConv(mid_channels, mid_channels)
        
        self.gnn_fc = nn.Linear(mid_channels, mid_channels)
        # -------------------
        # Classifier
        # -------------------
        self.classifier = nn.Sequential(
            nn.Linear(mid_channels, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.backbone(x)
    
        x = self.conv1(x)
        x = self.block1(x)
    
        x = self.conv2(x)
        x = self.block2(x)
    
        x = self.cbam(x)
        x = self.conv3(x)
        x = self.block3(x)
    
        # --------------------
        # CNN → Graph
        # --------------------
        B, C, H, W = x.shape
        nodes = x.view(B, C, H*W).permute(0, 2, 1)  # [B, N, C]
    
        adj = build_grid_adj(H, W, x.device)
    
        nodes = self.gnn1(nodes, adj)
        nodes = self.gnn2(nodes, adj)
    
        # Graph pooling
        gnn_feat = nodes.mean(dim=1)   # [B, C]
        gnn_feat = self.gnn_fc(gnn_feat)
    
        # --------------------
        # Final classifier
        # --------------------
        logits = self.classifier(gnn_feat)
        return logits


In [2]:
EPOCHS = 100                # لا تحتاج 200
BATCH_SIZE = 32             # لو GPU يسمح
LR = 3e-4                   # أعلى قليلًا للبداية السريعة
WEIGHT_DECAY = 1e-4
PATIENCE = 15               # EarlyStopping
FREEZE_EPOCHS = 10          # تجميد EfficientNet
#MIX_PROB = 0.2              # MixUp + CutMix
LABEL_SMOOTHING = 0.1
mixup_alpha = 0.2
mixup_prob  = 0.3
cutmix_prob = 0.0   # ❌ ممنوع

In [3]:
def freeze_backbone(model):
    for p in model.backbone.parameters():
        p.requires_grad = False

def unfreeze_backbone(model):
    for p in model.backbone.parameters():
        p.requires_grad = True


In [4]:
class EarlyStopping:
    def __init__(self, patience=15):
        self.patience = patience
        self.counter = 0
        self.best_acc = 0

    def step(self, val_acc):
        if val_acc > self.best_acc:
            self.best_acc = val_acc
            self.counter = 0
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience


In [5]:
def set_seed(seed=42):
    import random, numpy as np, torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)


In [6]:
  import os, re
from collections import Counter
from PIL import Image
from torch.utils.data import Dataset
import os
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler
from torchvision import transforms

class KDEFDataset(Dataset):
    # كود التعبير في اسم الملف -> اسم الكلاس
    EXPR = {'AN':'angry', 'DI':'disgust', 'AF':'fear', 'HA':'happy',
            'NE':'neutral', 'SA':'sad', 'SU':'surprise'}
    EMOTIONS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

    # AF01ANS.JPG -> session=A, subject=F01, expr=AN, angle=S
    PATTERN = re.compile(r'^([AB])([FM]\d{2})(AN|DI|AF|HA|NE|SA|SU)(S|HL|HR|FL|FR)\.(JPG|JPEG|PNG)$', re.I)

    def __init__(self, root_dir, transform=None, angles=None):
        """
        angles=None  -> كل الزوايا (4900 صورة تقريبًا)
        angles=['S'] -> الصور الأمامية بس (980 صورة)
        """
        self.transform = transform
        self.samples, self.subjects = [], []
        skipped = []
        for folder in sorted(os.listdir(root_dir)):
            fdir = os.path.join(root_dir, folder)
            if not os.path.isdir(fdir):
                continue
            for f in sorted(os.listdir(fdir)):
                m = self.PATTERN.match(f)
                if not m:
                    skipped.append(f); continue
                session, subj, expr, angle = m.group(1), m.group(2).upper(), m.group(3).upper(), m.group(4).upper()
                if angles and angle not in angles:
                    continue
                label = self.EMOTIONS.index(self.EXPR[expr])
                self.samples.append((os.path.join(fdir, f), label))
                self.subjects.append(subj)          # F01 / M17 ... (من غير الـ session)
        print(f"📦 Total samples : {len(self.samples)}")
        print(f"👤 Total subjects: {len(set(self.subjects))}")   # المفروض 70
        print(f"📊 Per class     : {Counter(self.EMOTIONS[l] for _, l in self.samples)}")
        if skipped:
            print(f"⚠️ Skipped {len(skipped)} files (name didn't match), e.g. {skipped[:5]}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

In [7]:
# ── Transforms ──────────────────────────────────────────────────────────────
from torchvision import transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

In [8]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import numpy as np

def train_one_fold(model, train_loader, test_loader,
                   fold_num, device,
                   num_epochs=50, patience=15):
    """
    يدرّب ويقيّم fold واحد.
    Returns: best_acc, best_f1, all_preds, all_labels
    """
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

    best_acc     = 0.0
    best_f1      = 0.0
    best_preds   = []
    best_labels  = []
    no_improve   = 0

    print(f"\n{'='*50}")
    print(f"  Fold {fold_num} — Training")
    print(f"{'='*50}")

    for epoch in range(1, num_epochs + 1):

        # ── Train ────────────────────────────────────────
        model.train()
        train_loss, correct, total = 0.0, 0, 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss    = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item() * imgs.size(0)
            preds       = outputs.argmax(dim=1)
            correct    += (preds == labels).sum().item()
            total      += imgs.size(0)

        scheduler.step()
        train_acc  = correct / total
        train_loss = train_loss / total

        # ── Evaluate ─────────────────────────────────────
        model.eval()
        all_preds, all_labels = [], []

        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs   = imgs.to(device)
                outputs = model(imgs)
                preds   = outputs.argmax(dim=1).cpu().tolist()
                all_preds  .extend(preds)
                all_labels .extend(labels.tolist())

        acc = accuracy_score(all_labels, all_preds) * 100
        f1  = f1_score(all_labels, all_preds,
                       average='weighted') * 100

        print(f"Epoch {epoch:03d}/{num_epochs} | "
              f"Loss={train_loss:.4f} | "
              f"TrainAcc={train_acc*100:.2f}% | "
              f"TestAcc={acc:.2f}% | F1={f1:.2f}%")

        # ── Early Stopping ────────────────────────────────
        if acc > best_acc:
            best_acc    = acc
            best_f1     = f1
            best_preds  = all_preds.copy()
            best_labels = all_labels.copy()
            no_improve  = 0
            # احفظ أحسن weights
            torch.save(model.state_dict(),
                       f'best_fold{fold_num}.pt')
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  ⏹ Early stopping at epoch {epoch}")
                break

    print(f"\n  ✅ Fold {fold_num} Best Acc = {best_acc:.2f}% | "
          f"F1 = {best_f1:.2f}%")

    return best_acc, best_f1, best_preds, best_labels

In [9]:
"""
JAFFE - subject-independent evaluation of DRAG-Net
==================================================
Default protocol : Leave-One-Subject-Out (LOSO) -> 10 folds, 1 test subject per fold
                   (the protocol used by the JAFFE subject-independent literature:
                    BDBN, Kas et al., Sun et al., Wadhawan & Gandhi, Baygin et al.).
Alternative      : PROTOCOL = "sgkf5" -> 5-fold StratifiedGroupKFold (2 test subjects per fold).

Needs from your notebook: FERModel, train_one_fold, train_transform, test_transform
(train_one_fold must use the JAFFE learning rate reported in Table 4: 1e-4).
"""
import os
import re
import json
import random
from collections import Counter

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, SubsetRandomSampler, Dataset
from sklearn.model_selection import LeaveOneGroupOut, StratifiedGroupKFold
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, precision_recall_fscore_support

# ============================================================
# Configuration (keep these identical to Table 4 of the paper)
# ============================================================
ROOT_DIR    = 'D/jaffeall/New Data'   # <-- your JAFFE folder
PROTOCOL    = 'loso'        # 'loso' (10 folds) or 'sgkf5' (5 folds)
SEED        = 42
BATCH_SIZE  = 16            # Table 4: JAFFE batch size
NUM_EPOCHS  = 50            # Table 4: JAFFE epochs (change here AND in Table 4 if you use another value)
PATIENCE    = NUM_EPOCHS    # = no early stopping (fixed number of epochs, as stated in the paper)
NUM_CLASSES = 7
EXPECTED_N  = 213
OUT_PREFIX  = f'jaffe_{PROTOCOL}'

JAFFE_EXTS    = ('.tiff', '.tif', '.jpg', '.jpeg', '.png', '.bmp')
JAFFE_CLASSES = ['AN', 'DI', 'FE', 'HA', 'NE', 'SA', 'SU']   # angry, disgust, fear, happy, neutral, sad, surprise
CLASS_TO_IDX  = {c: i for i, c in enumerate(JAFFE_CLASSES)}
_JAFFE_RE     = re.compile(r'^([A-Z]{2})\.([A-Z]{2})\d*\.', re.IGNORECASE)   # KA.AN1.39.tiff


# ============================================================
# Helpers
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_subject_jaffe(path):
    """KA.AN1.39.tiff -> KA  (10 subjects: KA KL KM KR MK NA NM TM UY YM)."""
    m = _JAFFE_RE.match(os.path.basename(path))
    if m is None:
        raise ValueError(f"Can't extract subject from: {path}")
    return m.group(1).upper()


def get_label_jaffe(path):
    """Emotion code from the file name (KA.AN1.39 -> AN); falls back to the parent folder name."""
    m = _JAFFE_RE.match(os.path.basename(path))
    code = m.group(2).upper() if m else None
    if code not in CLASS_TO_IDX:
        folder = os.path.basename(os.path.dirname(path)).upper()
        code = next((c for c in JAFFE_CLASSES if folder.startswith(c)), None)
    if code is None:
        raise ValueError(f"Can't extract emotion from: {path}")
    return CLASS_TO_IDX[code]


class JAFFEDataset(Dataset):
    """Recursive scan of the JAFFE folder; stores (path, label) pairs in a fixed (sorted) order."""
    def __init__(self, root_dir, transform=None):
        paths = sorted(os.path.join(d, f) for d, _, fs in os.walk(root_dir)
                       for f in fs if f.lower().endswith(JAFFE_EXTS))
        self.samples   = [(p, get_label_jaffe(p)) for p in paths]
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        p, y = self.samples[i]
        img = Image.open(p).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, y


def make_splits(labels, groups):
    idx = np.arange(len(labels))
    if PROTOCOL == 'loso':
        return list(LeaveOneGroupOut().split(idx, labels, groups=groups))
    if PROTOCOL == 'sgkf5':
        sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
        return list(sgkf.split(idx, labels, groups=groups))
    raise ValueError(f"Unknown PROTOCOL: {PROTOCOL}")


# ============================================================
# Main
# ============================================================
def run_subject_independent_jaffe(root_dir, device):
    set_seed(SEED)
    train_dataset = JAFFEDataset(root_dir, transform=train_transform)
    test_dataset  = JAFFEDataset(root_dir, transform=test_transform)
    assert [s[0] for s in train_dataset.samples] == [s[0] for s in test_dataset.samples], \
        "train/test datasets have a different sample order!"

    all_paths  = [p for p, _ in train_dataset.samples]
    all_labels = [l for _, l in train_dataset.samples]
    groups     = [get_subject_jaffe(p) for p in all_paths]
    subjects   = sorted(set(groups))

    print(f"📦 Total samples : {len(all_labels)}   (JAFFE should have {EXPECTED_N})")
    print(f"👤 Subjects      : {len(subjects)} {subjects}   (JAFFE should have 10)")
    print(f"🎭 Classes       : { {JAFFE_CLASSES[k]: v for k, v in sorted(Counter(all_labels).items())} }")
    print(f"🧪 Protocol      : {PROTOCOL}  | batch={BATCH_SIZE}  epochs={NUM_EPOCHS}  seed={SEED}")
    if len(all_labels) != EXPECTED_N:
        print(f"⚠️  expected {EXPECTED_N} images but got {len(all_labels)} - check the folder")
    assert len(subjects) == 10, "JAFFE must contain 10 subjects - check the file names"

    splits = make_splits(all_labels, groups)
    n_folds = len(splits)

    rows = []
    oof_preds, oof_labels = [], []

    for fold, (train_idx, test_idx) in enumerate(splits, start=1):
        set_seed(SEED)                                   # same seed for every fold -> reproducible
        train_s = sorted({groups[i] for i in train_idx})
        test_s  = sorted({groups[i] for i in test_idx})
        assert set(train_s).isdisjoint(test_s), f"Subject leakage in fold {fold}!"

        test_counts = {JAFFE_CLASSES[k]: v for k, v in sorted(Counter(all_labels[i] for i in test_idx).items())}
        print(f"\n{'━'*70}")
        print(f"  FOLD {fold}/{n_folds} | Train={len(train_idx)} ({len(train_s)} subj) | "
              f"Test={len(test_idx)} (subject(s): {', '.join(test_s)})")
        print(f"  Test class counts: {test_counts}")
        print(f"{'━'*70}")

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                                  sampler=SubsetRandomSampler(train_idx),
                                  num_workers=2, pin_memory=True,
                                  drop_last=True)          # avoids a 1-sample last batch (BatchNorm1d)
        test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                                  sampler=SubsetRandomSampler(test_idx),
                                  num_workers=2, pin_memory=True)

        model = FERModel(num_classes=NUM_CLASSES).to(device)
        acc_ret, _, preds, labels = train_one_fold(model, train_loader, test_loader,
                                                   fold_num=fold, device=device,
                                                   num_epochs=NUM_EPOCHS, patience=PATIENCE)
        preds  = np.asarray(preds).ravel().astype(int)
        labels = np.asarray(labels).ravel().astype(int)
        assert len(labels) == len(test_idx), "train_one_fold returned a different number of test samples"

        acc = accuracy_score(labels, preds) * 100
        mf1 = f1_score(labels, preds, average='macro', labels=list(range(NUM_CLASSES)), zero_division=0) * 100
        wf1 = f1_score(labels, preds, average='weighted', zero_division=0) * 100
        if abs(acc - float(acc_ret)) > 0.01:
            print(f"  ⚠️  accuracy returned by train_one_fold ({float(acc_ret):.2f}%) differs from the "
                  f"accuracy of the returned predictions ({acc:.2f}%)")

        rows.append({'Fold': fold, 'Test subject(s)': ', '.join(test_s), 'Test images': len(test_idx),
                     'Accuracy (%)': round(acc, 2), 'Macro F1 (%)': round(mf1, 2), 'Weighted F1 (%)': round(wf1, 2)})
        oof_preds.extend(preds.tolist())
        oof_labels.extend(labels.tolist())
        print(f"  ➜ Fold {fold}: Acc={acc:.2f}%  MacroF1={mf1:.2f}%  WeightedF1={wf1:.2f}%")

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ---------------- summary ----------------
    df = pd.DataFrame(rows)
    accs, mf1s, wf1s = df['Accuracy (%)'].values, df['Macro F1 (%)'].values, df['Weighted F1 (%)'].values

    cm = confusion_matrix(oof_labels, oof_preds, labels=list(range(NUM_CLASSES)))
    oof_acc = accuracy_score(oof_labels, oof_preds) * 100
    p, r, f, _ = precision_recall_fscore_support(oof_labels, oof_preds, average='macro',
                                                 labels=list(range(NUM_CLASSES)), zero_division=0)
    recall_per_class = cm.diagonal() / cm.sum(axis=1) * 100

    title = 'LEAVE-ONE-SUBJECT-OUT' if PROTOCOL == 'loso' else '5-FOLD SUBJECT-INDEPENDENT'
    print(f"\n{'═'*70}")
    print(f"  {title} RESULTS — JAFFE ({n_folds} folds)")
    print(f"{'═'*70}")
    print(df.to_string(index=False))
    print(f"{'─'*70}")
    # population SD (ddof=0), same convention as Tables 7b and 9b
    print(f"  Mean Acc (over folds)  : {accs.mean():.2f}% ± {accs.std():.2f}%")
    print(f"  Mean Macro F1          : {mf1s.mean():.2f}% ± {mf1s.std():.2f}%")
    print(f"  Mean Weighted F1       : {wf1s.mean():.2f}% ± {wf1s.std():.2f}%")
    print(f"  Pooled (OOF) accuracy  : {oof_acc:.2f}%  ({int(cm.trace())}/{int(cm.sum())})")
    print(f"  Pooled macro P / R / F1: {p*100:.2f}% / {r*100:.2f}% / {f*100:.2f}%")
    print(f"{'═'*70}")

    cm_df = pd.DataFrame(cm, index=JAFFE_CLASSES, columns=JAFFE_CLASSES)
    cm_df['Recall (%)'] = np.round(recall_per_class, 2)
    print("\n  Confusion matrix (all folds combined; rows = actual, columns = predicted):")
    print(cm_df.to_string())

    # ---------------- save ----------------
    df.to_csv(f'{OUT_PREFIX}_per_fold.csv', index=False)
    cm_df.to_csv(f'{OUT_PREFIX}_confusion_matrix.csv')
    summary = {'protocol': PROTOCOL, 'folds': n_folds, 'seed': SEED,
               'batch_size': BATCH_SIZE, 'epochs': NUM_EPOCHS,
               'mean_acc': round(float(accs.mean()), 2), 'sd_acc': round(float(accs.std()), 2),
               'mean_macro_f1': round(float(mf1s.mean()), 2), 'sd_macro_f1': round(float(mf1s.std()), 2),
               'mean_weighted_f1': round(float(wf1s.mean()), 2), 'sd_weighted_f1': round(float(wf1s.std()), 2),
               'oof_acc': round(float(oof_acc), 2),
               'oof_macro_precision': round(float(p * 100), 2),
               'oof_macro_recall': round(float(r * 100), 2),
               'oof_macro_f1': round(float(f * 100), 2)}
    with open(f'{OUT_PREFIX}_summary.json', 'w') as fh:
        json.dump(summary, fh, indent=2)
    print(f"\n💾 Saved: {OUT_PREFIX}_per_fold.csv, {OUT_PREFIX}_confusion_matrix.csv, {OUT_PREFIX}_summary.json")

    return {'per_fold': df, 'confusion_matrix': cm_df, 'summary': summary}


if __name__ == "__main__":
    device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    results = run_subject_independent_jaffe('D/jaffeall/New Data', device)

📦 Total samples : 213   (JAFFE should have 213)
👤 Subjects      : 10 ['KA', 'KL', 'KM', 'KR', 'MK', 'NA', 'NM', 'TM', 'UY', 'YM']   (JAFFE should have 10)
🎭 Classes       : {'AN': 30, 'DI': 29, 'FE': 32, 'HA': 31, 'NE': 30, 'SA': 31, 'SU': 30}
🧪 Protocol      : loso  | batch=16  epochs=100  seed=42

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  FOLD 1/10 | Train=190 (9 subj) | Test=23 (subject(s): KA)
  Test class counts: {'AN': 3, 'DI': 3, 'FE': 4, 'HA': 4, 'NE': 3, 'SA': 3, 'SU': 3}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Fold 1 — Training
Epoch 001/100 | Loss=2.0216 | TrainAcc=17.05% | TestAcc=13.04% | F1=3.01%
Epoch 002/100 | Loss=1.8712 | TrainAcc=22.73% | TestAcc=13.04% | F1=3.01%
Epoch 003/100 | Loss=1.7974 | TrainAcc=28.41% | TestAcc=13.04% | F1=3.26%
Epoch 004/100 | Loss=1.7733 | TrainAcc=28.98% | TestAcc=26.09% | F1=12.01%
Epoch 005/100 | Loss=1.7336 | TrainAcc=35.80% | TestAcc=39.13% | F1=22.77%
Epoch 006/100 | Los